In [3]:
from functools import total_ordering

import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv("./data/netflix-699bb0d94f78e745148155.csv")

### Data exploration

In [5]:
nrows,ncols = df.shape
print(f'the data fram has {nrows} rows and {ncols} columns')
df.head(10)

the data fram has 8807 rows and 12 columns


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
5,s6,TV Show,Midnight Mass,Mike Flanagan,"Kate Siegel, Zach Gilford, Hamish Linklater, H...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",The arrival of a charismatic young priest brin...
6,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, ...",NaN,"September 24, 2021",2021,PG,91 min,Children & Family Movies,Equestria's divided. But a bright-eyed hero be...
7,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...","United States, Ghana, Burkina Faso, United Kin...","September 24, 2021",1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s..."
8,s9,TV Show,The Great British Baking Show,Andy Devonshire,"Mel Giedroyc, Sue Perkins, Mary Berry, Paul Ho...",United Kingdom,"September 24, 2021",2021,TV-14,9 Seasons,"British TV Shows, Reality TV",A talented batch of amateur bakers face off in...
9,s10,Movie,The Starling,Theodore Melfi,"Melissa McCarthy, Chris O'Dowd, Kevin Kline, T...",United States,"September 24, 2021",2021,PG-13,104 min,"Comedies, Dramas",A woman adjusting to life after a loss contend...


In [6]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='str')

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [8]:
df["date_added"] = pd.to_datetime(df["date_added"],format="mixed")

In [9]:
#No data is duplicated
print('duplicated ids :',df.duplicated(subset=['title']).sum())
print('duplicated titles :',df.duplicated(subset=['title']).sum())


duplicated ids : 0
duplicated titles : 0


## Null Values

In [10]:
#Presence of null values
print(df.isnull().sum())
print("there are null values especially in director,cast and country")

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64
there are null values especially in director,cast and country


### description column is not usful for analysis

In [11]:
df = df.drop(columns=["description"])

### lets start with the null value easiest to treat (duration, rating)

In [12]:
df[df["duration"].isnull()]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in
5541,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,2017-04-04,2017,74 min,NaN,Movies
5794,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,2016-09-16,2010,84 min,NaN,Movies
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,2016-08-15,2015,66 min,NaN,Movies


### Some how the duration seemed to have leaked into the rating column for the louis C. K. movies

In [13]:
#Replace the null values with the ratings values
df["duration"] = df["duration"].fillna(df["rating"])

In [14]:
df["rating"].value_counts().keys()

Index(['TV-MA', 'TV-14', 'TV-PG', 'R', 'PG-13', 'TV-Y7', 'TV-Y', 'PG', 'TV-G',
       'NR', 'G', 'TV-Y7-FV', 'NC-17', 'UR', '74 min', '84 min', '66 min'],
      dtype='str', name='rating')

#### min values cannot be considered as ratings so they must be removed

In [15]:
valid_ratings = ['TV-MA', 'TV-14', 'TV-PG', 'R', 'PG-13', 'TV-Y7', 'TV-Y', 'PG', 'TV-G',
       'NR', 'G', 'TV-Y7-FV', 'NC-17', 'UR']

def validate_rating(rating):
    if rating not in valid_ratings:
        return np.nan
    else:
        return rating


df["rating"] = df["rating"].apply(validate_rating)
print("now the df contains ",df["rating"].isnull().sum()," null ratings")

now the df contains  7  null ratings


In [16]:
df[df["rating"].isnull()]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in
5541,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,2017-04-04,2017,NaN,74 min,Movies
5794,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,2016-09-16,2010,NaN,84 min,Movies
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,2016-08-15,2015,NaN,66 min,Movies
5989,s5990,Movie,13TH: A Conversation with Oprah Winfrey & Ava ...,NaN,"Oprah Winfrey, Ava DuVernay",NaN,2017-01-26,2017,NaN,37 min,Movies
6827,s6828,TV Show,Gargantia on the Verdurous Planet,NaN,"Kaito Ishikawa, Hisako Kanemoto, Ai Kayano, Ka...",Japan,2016-12-01,2013,NaN,1 Season,"Anime Series, International TV Shows"
7312,s7313,TV Show,Little Lunch,NaN,"Flynn Curry, Olivia Deeble, Madison Lu, Oisín ...",Australia,2018-02-01,2015,NaN,1 Season,"Kids' TV, TV Comedies"
7537,s7538,Movie,My Honor Was Loyalty,Alessandro Pepe,"Leone Frisa, Paolo Vaccarino, Francesco Miglio...",Italy,2017-03-01,2015,NaN,115 min,Dramas


### We can manually replace the missing values since there are only 7

In [17]:
ratings_dict = {
    "Louis C.K. 2017": "TV-MA",
    "Louis C.K.: Hilarious": "TV-MA",
    "Louis C.K.: Live at the Comedy Store": "TV-MA",
    "13TH: A Conversation with Oprah Winfrey & Ava DuVernay": "TV-14",
    "Gargantia on the Verdurous Planet": "TV-14",
    "Little Lunch": "TV-G",
    "My Honor Was Loyalty": "NR"
}

for key, value in ratings_dict.items():
    df.loc[df["title"] == key,"rating"] = value

In [18]:
print(df.isnull().sum())

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             0
duration           0
listed_in          0
dtype: int64


### we can drop the date added null columns since it is not significant enaugh to effect the goal results

In [19]:
df = df.dropna(subset=['date_added'])
print(df.isnull().sum())

show_id            0
type               0
title              0
director        2624
cast             825
country          830
date_added         0
release_year       0
rating             0
duration           0
listed_in          0
dtype: int64


### we can replace null cast values with empty string since it is a list of actors in the films/shows and it is not in our intrest

In [20]:
df["cast"] = df["cast"].fillna("")

### we need all countries to be listed since we need them for analysis there for we delete the nulls

In [21]:
df = df.dropna(subset=["country"])

### We dont need director for our analysis but we cant delete it since the null are significant so we drop the column

In [22]:
df = df.drop(columns=["director"])

In [23]:
print(df.isnull().sum())

show_id         0
type            0
title           0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64


## Transforming Multi-value columns

### lets process the listed in column

In [24]:
df["listed_in"].value_counts().index

Index(['Documentaries', 'Dramas, International Movies', 'Stand-Up Comedy',
       'Comedies, Dramas, International Movies',
       'Dramas, Independent Movies, International Movies',
       'Children & Family Movies, Comedies', 'Kids' TV',
       'Documentaries, International Movies',
       'Dramas, International Movies, Romantic Movies',
       'Comedies, International Movies',
       ...
       'Children & Family Movies, Faith & Spirituality',
       'Classic Movies, Comedies, Sports Movies',
       'Comedies, Dramas, Sports Movies',
       'Action & Adventure, Romantic Movies, Sci-Fi & Fantasy',
       'Classic & Cult TV, TV Sci-Fi & Fantasy',
       'Comedies, Cult Movies, LGBTQ Movies',
       'Action & Adventure, Comedies, Horror Movies',
       'Classic & Cult TV, Crime TV Shows, TV Dramas',
       'Action & Adventure, Documentaries, Sports Movies',
       'Cult Movies, Dramas, Thrillers'],
      dtype='str', name='listed_in', length=497)

In [25]:
all_listings = df["listed_in"].str.split(", ")
print(len(all_listings))
all_listings.head()

7967


0                                      [Documentaries]
1    [International TV Shows, TV Dramas, TV Mysteries]
4    [International TV Shows, Romantic TV Shows, TV...
7    [Dramas, Independent Movies, International Mov...
8                       [British TV Shows, Reality TV]
Name: listed_in, dtype: object

In [26]:
scrambled_genres = all_listings.value_counts().index
genres = set()
for genre_list in scrambled_genres:
    for genre in genre_list:
        genres.add(genre.strip())

genres = list(genres)
print(len(genres))
genres


42


['TV Dramas',
 'Sports Movies',
 'British TV Shows',
 'Children & Family Movies',
 'TV Comedies',
 'Korean TV Shows',
 'Docuseries',
 'Science & Nature TV',
 'Horror Movies',
 'TV Shows',
 'Documentaries',
 'Independent Movies',
 'Classic & Cult TV',
 'Dramas',
 'Comedies',
 'LGBTQ Movies',
 'Teen TV Shows',
 'Music & Musicals',
 'Action & Adventure',
 'Spanish-Language TV Shows',
 'Anime Series',
 'Romantic Movies',
 'Romantic TV Shows',
 'Reality TV',
 'Sci-Fi & Fantasy',
 'International Movies',
 'TV Mysteries',
 'Faith & Spirituality',
 'TV Thrillers',
 'TV Horror',
 'Crime TV Shows',
 'Stand-Up Comedy & Talk Shows',
 'TV Action & Adventure',
 'TV Sci-Fi & Fantasy',
 'Cult Movies',
 'Movies',
 'Stand-Up Comedy',
 "Kids' TV",
 'Classic Movies',
 'Anime Features',
 'Thrillers',
 'International TV Shows']

In [27]:
del(all_listings)

In [28]:
genre_counts_df = df["listed_in"].str.split(", ").explode().value_counts().reset_index()
genre_counts_df.columns = ['Genre', 'Count']
genre_counts_df.head()

,Genre,Count
0,International Movies,2543
1,Dramas,2317
2,Comedies,1580
3,International TV Shows,1127
4,Action & Adventure,817


### lets process the country in column

In [29]:
df["country"].value_counts().index

Index(['United States', 'India', 'United Kingdom', 'Japan', 'South Korea',
       'Canada', 'Spain', 'France', 'Mexico', 'Egypt',
       ...
       'Germany, United States, United Kingdom, Canada',
       'Canada, India, Thailand, United States, United Arab Emirates',
       'United States, East Germany, West Germany',
       'France, Netherlands, South Africa, Finland',
       'Egypt, Austria, United States', 'Russia, Spain',
       'Croatia, Slovenia, Serbia, Montenegro', 'Japan, Canada',
       'United States, France, South Korea, Indonesia',
       'United Arab Emirates, Jordan'],
      dtype='str', name='country', length=748)

In [30]:
all_countrie_listings = df["country"].str.split(", ")

In [31]:
scrambled_countries = all_countrie_listings.value_counts().index
del(all_countrie_listings)
countries = set()
for country_list in scrambled_countries:
    for countrie in country_list:
        countries.add(countrie.strip())

countries = list(countries)
print(len(countries))
countries


127


['',
 'Finland',
 'Australia',
 'Serbia',
 'Uruguay',
 'Cuba',
 'Algeria',
 'Philippines',
 'Puerto Rico',
 'Malaysia',
 'Botswana',
 'Sudan',
 'Belgium',
 'Cambodia,',
 'Belarus',
 'Netherlands',
 'Sri Lanka',
 'Bangladesh',
 'Vietnam',
 'Georgia',
 'Qatar',
 'Thailand',
 'Venezuela',
 'Peru',
 'Slovakia',
 'Austria',
 'Singapore',
 'South Africa',
 'Guatemala',
 'Indonesia',
 'Ukraine',
 'Azerbaijan',
 'Switzerland',
 'Brazil',
 'Poland,',
 'Germany',
 'Jordan',
 'Somalia',
 'France',
 'Saudi Arabia',
 'United States',
 'Burkina Faso',
 'Cameroon',
 'Hungary',
 'Nicaragua',
 'Jamaica',
 'Italy',
 'Angola',
 'Nigeria',
 'Pakistan',
 'Chile',
 'Bulgaria',
 'Iceland',
 'Egypt',
 'Spain',
 'China',
 'Norway',
 'Sweden',
 'Bermuda',
 'Romania',
 'Taiwan',
 'Japan',
 'Cambodia',
 'Slovenia',
 'Bahamas',
 'Malawi',
 'Namibia',
 'Ecuador',
 'Lebanon',
 'India',
 'Cyprus',
 'Greece',
 'Argentina',
 'Ethiopia',
 'United Arab Emirates',
 'United States,',
 'Hong Kong',
 'Vatican City',
 'Uganda

In [32]:
country_pivot_df = df["country"].str.split(', ').explode().value_counts().reset_index()
country_pivot_df.head()

,country,count
0,United States,3683
1,India,1046
2,United Kingdom,803
3,Canada,445
4,France,393


### the cast

In [33]:
df.head()

,show_id,type,title,cast,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,,United States,2021-09-25,2020,PG-13,90 min,Documentaries
1,s2,TV Show,Blood & Water,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries"
4,s5,TV Show,Kota Factory,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ..."
7,s8,Movie,Sankofa,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...","United States, Ghana, Burkina Faso, United Kin...",2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"
8,s9,TV Show,The Great British Baking Show,"Mel Giedroyc, Sue Perkins, Mary Berry, Paul Ho...",United Kingdom,2021-09-24,2021,TV-14,9 Seasons,"British TV Shows, Reality TV"


In [34]:
cast_pivot_df = df["cast"].str.split(', ').explode().str.strip().loc[lambda x: x != ""].value_counts().reset_index()
cast_pivot_df

,cast,count
0,Anupam Kher,43
1,Shah Rukh Khan,34
2,Naseeruddin Shah,31
3,Akshay Kumar,30
4,Om Puri,30
...,...,...
34281,Ryan Newman,1
34282,Raaghav Chanana,1
34283,Malkeet Rauni,1
34284,Anita Shabdish,1


### Date split

In [35]:
df["Year_added"] = df["date_added"].dt.year
df["Month"] = df["date_added"].dt.month
df["Day"] = df["date_added"].dt.day

In [36]:
df.head()

,show_id,type,title,cast,country,date_added,release_year,rating,duration,listed_in,Year_added,Month,Day
0,s1,Movie,Dick Johnson Is Dead,,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,9,25
1,s2,TV Show,Blood & Water,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries",2021,9,24
4,s5,TV Show,Kota Factory,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",2021,9,24
7,s8,Movie,Sankofa,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...","United States, Ghana, Burkina Faso, United Kin...",2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,9,24
8,s9,TV Show,The Great British Baking Show,"Mel Giedroyc, Sue Perkins, Mary Berry, Paul Ho...",United Kingdom,2021-09-24,2021,TV-14,9 Seasons,"British TV Shows, Reality TV",2021,9,24


### Exporting the data

In [37]:
with pd.ExcelWriter("./data/clean_data.xlsx", engine="xlsxwriter") as writer:
    df.to_excel(writer, sheet_name="Netflix", index=False)
    genre_counts_df.to_excel(writer, sheet_name="Genre Pivot", index=False)
    country_pivot_df.to_excel(writer, sheet_name="Country Pivot", index=False)
    cast_pivot_df.to_excel(writer, sheet_name="Cast Pivot", index=False)



In [38]:
del(df,genre_counts_df,country_pivot_df,ratings_dict,countries,genres,scrambled_countries,scrambled_genres,cast_pivot_df)

In [39]:
del(writer,valid_ratings)

### Explorations

In [40]:
cleaned_df = pd.read_excel("./data/clean_data.xlsx",sheet_name="Netflix")

In [41]:
cleaned_df.shape

(7967, 13)

In [4]:
cleaned_df.columns

Index(['show_id', 'type', 'title', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'Year_added',
       'Month', 'Day'],
      dtype='str')

In [5]:
cleaned_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7967 entries, 0 to 7966
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       7967 non-null   str           
 1   type          7967 non-null   str           
 2   title         7967 non-null   str           
 3   cast          7296 non-null   str           
 4   country       7967 non-null   str           
 5   date_added    7967 non-null   datetime64[us]
 6   release_year  7967 non-null   int64         
 7   rating        7967 non-null   str           
 8   duration      7967 non-null   str           
 9   listed_in     7967 non-null   str           
 10  Year_added    7967 non-null   int64         
 11  Month         7967 non-null   int64         
 12  Day           7967 non-null   int64         
dtypes: datetime64[us](1), int64(4), str(8)
memory usage: 809.3 KB


In [56]:
genre_pivot_df = pd.read_excel("./data/clean_data.xlsx",sheet_name="Genre Pivot")
genre_pivot_df.head()

,Genre,Count
0,International Movies,2543
1,Dramas,2317
2,Comedies,1580
3,International TV Shows,1127
4,Action & Adventure,817


In [57]:
genre_pivot_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Genre   42 non-null     str  
 1   Count   42 non-null     int64
dtypes: int64(1), str(1)
memory usage: 804.0 bytes


In [58]:
country_pivot_df = pd.read_excel("./data/clean_data.xlsx",sheet_name="Country Pivot")
country_pivot_df.head()

,country,count
0,United States,3683
1,India,1046
2,United Kingdom,803
3,Canada,445
4,France,393


In [59]:
country_pivot_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 127 entries, 0 to 126
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   country  126 non-null    str  
 1   count    127 non-null    int64
dtypes: int64(1), str(1)
memory usage: 2.1 KB


In [60]:
cleaned_df["rating"].value_counts().head()

rating
TV-MA    2932
TV-14    1928
R         788
TV-PG     771
PG-13     482
Name: count, dtype: int64

In [42]:
cast_pivot_df = pd.read_excel("./data/clean_data.xlsx",sheet_name="Cast Pivot")
cast_pivot_df.head()

,cast,count
0,Anupam Kher,43
1,Shah Rukh Khan,34
2,Naseeruddin Shah,31
3,Akshay Kumar,30
4,Om Puri,30


In [43]:
cast_pivot_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34286 entries, 0 to 34285
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   cast    34286 non-null  str  
 1   count   34286 non-null  int64
dtypes: int64(1), str(1)
memory usage: 535.8 KB
